In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pagerank_implementation import * 

# Section 4: Analysis & Experiments

In this section we will analyze the behaviour of the PageRank algorithm, validate the applied mathematical concepts, explore the actual aplication and test the robustness of the algorithm. To do so, we will perform four experiments as follows: 

1. Initial Vector Independence (Mathematical Foundation) - During this experiment we will demonstrate the strength of the PageRank algorithm - regardless of how the surfer starts its journey within the web, the pages score remains steady. We will test this by applying 3 different initial vectors, run Power Iteration from each starting point and verify that in all 3 cases result in the same final vector.
2. Personalized PageRank (Practical Application): in this experiment we are going to simulate personalized behaviour of the web surferer. We will consider that the surfer is biased and jumps towards specific pages. The question we will try to answer is how this personalized behavoir changes the page ranking to favor user interest? We will touch base the topic for applying user history and personalization for generating a page score.
3.  Network Connectivity (Structural Analysis): in this experiment we will investigate how sensitive is the network structure to changes and more specifically to removing links. Generally, the web structure determines the importance of the pages and their RangeRank score. We will show how the scores change and compare which pages are most affected when bridges between links are removed.
4.  Link Manipulation (vulnerability Testing)- in this last subsection we will demonstrate how adding artificial links in the web can boost a pages' ranking. This is the so called 'search engine optimization" attack and shows how vulnerable is PageRank to strategic and intentinal link manipulation.        

## Initial Vector Independence
This experiment aims to validate the main feature of the PageRank algorithm - the steady-state distribution of the scores is unique regardless of the starting point (initial probability distribution). [Refer to Perron-Frobenius Theorem]. We will execute the experiment by running the Power Iteration from 5 differnt starting points:

1. Uniform Distribution - all pages equally likely (already applied in the implementation phase Section 3)
2. Concentrated on Page A - all probability concentrated on one page.
3. 3 Random distributions - sampled from Dirichlet distribution [Refer to Dirichlet distribution why we choose it]

Doing the above 3 ponts we will show that the dominat eigenvector is unique for our primite stochastic matrices.

In [ ]:
d = 0.85 #damping_factor

n = m_prime_web.shape[0] #erturns the number of pages in the web

v_uniform = np.ones(n) / n # the uniform distribution vector - default vector used in Section 3 

v_pageA = np.zeros(n) # personalized vector with 1x8 (n) zeros
v_pageA[0] = 1 # all probability goes to page A - we visit only page A

We are going to use the Power Iteration funciton to show that regardless of the input vector, the web scores will be equal to the eigenvalues.

In [ ]:
def power_iteration_custom_vector(m_prime, v_initial, epsilon = 1e-6, max_iterations = 1000):
    x = v_initial.copy() # Just change the uniform distribution to personalized vector -> The rest is the same
    error_history = [] # Storage for convergence error tracking
    
    # Main iteration loop
    for iteration in range(max_iterations):
        x_new = m_prime @ x
        
        error = np.linalg.norm(x_new - x, ord=1)
        error_history.append(error)
        
        if error < epsilon: 
            print(f"Converged in {iteration + 1} with Final error = {error:.2e}\n")
            return x_new, error_history
        
        x = x_new # update vector
    
    print (f"Did not converge in {max_iterations} iterations")
    
    return x, error_history

In [ ]:
r_uni, history_uni = power_iteration_custom_vector(m_prime_web, v_uniform)

r_pageA, history_pageA = power_iteration_custom_vector(m_prime_web, v_pageA)

In [ ]:
print("Side-by-Side Comparison:")
print("-"*65)
print("Page | Uniform V  |  PageA V  |    EV    | Uni - EV | PageA - EV")
print("-"*65)
for i, label in enumerate(page_labels):
    diff_1 = abs(r_uni[i] - r_eig[i])
    diff_2 = abs(r_pageA[i] - r_eig[i])
    print(f"  {label}  |  {r_uni[i]:.6f}  |  {r_pageA[i]:.6f} | {r_pageA[i]:.6f} | {diff_1:.6f} | {diff_2:.6f}")


Results Match - both starting vectors converge to the same steady-state of the PageRank scores. However, to further test the steady-state of the web scores, we will test with random probability distributions.

We sample from the Dirichlet distribution [add reference]:
  * Dirichlet $(\alpha_1, \dots, \alpha_n)$ generates random probability vectors
  * With $\alpha = [1, 1, \dots, 1]$, we get uniform distribution over the
    probability simplex Δⁿ (all valid probability distributions
    are equally likely)
  * This gives us arbitrary, unbiased random starting points

In [ ]:
all_results = {
    'Uniform': (r_uni, history_uni),
    'Biased A': (r_pageA, history_pageA),
    'EigenValues': (r_eig,error)
} 

num_random = 3 # generate 3 random initial vectors

for i in range(num_random):
    alpha = np.ones(n)
    v_random = np.random.dirichlet(alpha)

    r_random, history_random = power_iteration_custom_vector(m_prime_web, v_random)

    label = f'Random_{i+1}'
    all_results[label] = (r_random, history_random)

In [ ]:
r_reference = all_results['EigenValues'][0] # use for value comparision 

print("L1 Distance from Uniform Result:")
print("-"*30)
for label, (r, _) in all_results.items():
    if label == 'Uniform':
        continue
    # Compute L1 norm of difference
    distance = np.linalg.norm(r - r_reference, ord=1)
    
    print(f"  {label:18s}: {distance:.6f}")

All 5 starting vectors (uniform, concentrated and 3 random vectors) converged to an identical steady-states. this once more. This empirically validates the Perron-Frobenius theorem's guarantee that primitive stochastic matrices have a unique dominant eigenvector.The steady-state doesn't depend on initial conditions - it's a property of the GRAPH STRUCTURE (matrix M'), not our arbitrary starting assumptions (vector v₀).

## Personalized PageRank

Currenly, for calculating the PageRank scores we apply the uniform distribution -  a random surfer jumps from pages to pages, so each webpage has "equal" probability to be visited. However, in reality, the surfer will visit the page that "favors" and the distribution is actually biased. We will represent this biased behaviour by modifiyng the transition matrix formula and including a personalization vector. We will test three types of "teleportations":

1. Standard ("uniform teleportation")
2. Personalized for Page A (always visit page A)
3. Personalized for Spider Trap F+G (favor those pages)

We will start with the standard PageRank computation where the surfer can jump to any page with equal probability - uniform distribution (1/n each). This was already done in the previous section, so we will show just the transformation matrix with applied damping effect and the resulted PageRank scores from applying the power iteration approach.

In [ ]:
m_web #transition matrix without damping
m_prime_web #transition matrix with damping
r_web  #pagerank values
history_web #convergence history
r_eig #eigenvalues

print("\nPage | PageRank Score | Percentage")
print("-" * 42)
for i, (label, score) in enumerate(zip(page_labels, r_web)):
    print(f"  {label}  |    {score:.6f}    |   {score*100:.2f}%")
print("-" * 42)
print(f"Sum: {r_web.sum():.6f} (should be 1.0)")

The original formula we applied for calculating the transition matrix was: $M' = d \cdot M + \frac{1 - d}{n} \cdot E$ 

Now we are going to apply the modified formula [reference here]: $M' = d \cdot M + (1 - d) \cdot v_{personalized} \cdot 1^T$;

where  $v_{personalized}$ is the biased vector.

Now we are going to add the bias effect where the surfer prefers to jumps towars page A by applying a personalized initial vector where the probability of visiting page A is increased to 1.0 (100%) -> $[1,0,0 \dots ,0]$


In [ ]:
d = 0.85 #damping_factor

transition_matrix = m_web #original transition matrix
n = transition_matrix.shape[0] #erturns the number of pages in the web

v_pageA = np.zeros(n) # personalized vector with 1x8 (n) zeros
v_pageA[0] = 1 # all probability goes to page A

personalization_matrix = np.zeros((n,n))
personalization_matrix[0,:] = np.ones(n) #all probability assigned to page A in the personalization matrix

In [ ]:
personalization_matrix = np.outer(v_pageA,np.ones(n)) # we can use np.outer for more complicated personalized matrix - returning the outer product of two vectors - Matrix (n x n)

In [ ]:
m_prime_pageA = d * transition_matrix + (1-d) * personalization_matrix
column_sums_pageA = m_prime_pageA.sum(axis = 0) 

print(f"Transition Matrix with biased behaviour\n")
print (f"{m_prime_pageA}\n")
print(f"Sum of Columns: {column_sums_pageA} (should be 1.0 for all columns)")

In [ ]:
#test with damping effect
transition_matrix = m_prime_web
n = transition_matrix.shape[0] #erturns the number of pages

v_pageA = np.zeros(n) # personalized vector
v_pageA[0] = 1 # all probability goes to page A

personalization_matrix = np.zeros((n,n))
personalization_matrix[0,:] = np.ones(n) #all probability assigned to page A in the personalization matrix

m_prime_pageA = d * transition_matrix + (1-d) * personalization_matrix
column_sums_pageA = m_prime_pageA.sum(axis = 0) 

print(f"Transition Matrix with biased behaviour\n")
print (f"{m_prime_pageA}\n")
print(f"Sum of Columns: {column_sums_pageA} (should be 1.0 for all columns)")

In [ ]:
#Define Personal Transition Matrix with Damping Effect 
def personal_transition_matrix (transition_matrix, personalization_vector, damping_factor = 0.85):
    n = transition_matrix.shape[0] #erturns the number of pages
    d = damping_factor

    personalization_matrix = np.outer(personalization_vector,np.ones(n)) #v_personalized * 1^T
    m_prime = d * transition_matrix + (1 - d) * personalization_matrix # apply modified scoring formula

    return m_prime

In [ ]:
m_prime_pageA = personal_transition_matrix(m_prime_web,v_pageA)

In [ ]:
print(f"Transition Matrix with biased behaviour\n")
print (f"{m_prime_pageA}\n")
print(f"Sum of Columns: {m_prime_pageA.sum(axis=0)} (should be 1.0 for all columns)")

In [ ]:
#Compare the Power Iteration and Eigenvector Score with the new transition matrix
r_web, history_web # PI Standard page scores
r_eig, error #Standard Eigenvector page scores

r_pi_pageA, history_pi_pageA = pagerank_power_iteration(m_prime_pageA,1e-6) # apply the personalized vector 

print("\nPage | Standard | Biased A  | Change_pi | Change_ev  ")
print("-"*50)
for i, label in enumerate(page_labels):
    change_pi = r_pi_pageA[i] - r_web[i]
    change_ev = r_pi_pageA[i] - r_eig[i]
    print(f"  {label}  | {r_eig[i]:.6f} | {r_pi_pageA[i]:.6f} | {change_pi:+.6f} | {change_ev:+.6f}")

In [ ]:
r_ev_pageA = find_ev(m_prime_pageA) # apply the personalized transition matrix

print("\nPage | Standard | Biased A  | Change_pi | Change_ev  ")
print("-"*50)
for i, label in enumerate(page_labels):
    change_pi = r_ev_pageA[i] - r_web[i]
    change_ev = r_ev_pageA[i] - r_eig[i]
    print(f"  {label}  | {r_eig[i]:.6f} | {r_ev_pageA[i]:.6f} | {change_pi:+.6f} | {change_ev:+.6f}")

From the above comparison we can see that the biased behavior of the surfer actually changes significantly the PageRank score of pageA. While, initialy the page was mostly never visited, now its score significantly increasd with 17% due to the application of the surfer preferences. Another interesting result is that favoring page A is actually affecting the overall score of the main cluster of pages in the - from A to E while it takes scores from the spider trap F-G. If we turn this and favor the spider-trap then we will espect pages F and G to dominate the web, besides being isolated. The dangling node H is stable during the scenarios.     

In [ ]:
#Test with Spider Trap Personalization
main_cluster = [0,1,2,3,4] #Main Pages A,B,C,D,E
spider_trap = [5,6] #F-G spider Trap

v_trap = np.zeros(n)

for i in spider_trap:
    v_trap[i] = 0.5 # Assing 50/50 probability to the pages in the spider trap F-G

m_prime_trap = personal_transition_matrix(m_web,v_trap)

r_trap = find_ev(m_prime_trap) 

In [ ]:
sorted_indices_trap = np.argsort(r_trap)[::-1]  # Descending order
for rank, idx in enumerate(sorted_indices_trap, 1):
    label = page_labels[idx]
    score = r_trap[idx]
    print(f"  Rank {rank}: Page {label}  -  {score:.6f}  ({score * 100:.2f}%)")

In [ ]:
#Standard vs Pers. A vs Pers. F+G
print("Page | Standard | Pers. A  |Trap F+G  |")
print("-"*40)
for i, label in enumerate(page_labels):  
    print(f"  {label}  | {r_eig[i]:.6f} | {r_ev_pageA[i]:.6f} | {r_trap[i]:.6f} | ")

From the above experiments we can conclude: 

1. Personalization behaviour matters:
   * Biasing toward Page A increases its score by 17%.
   * Biasing toward F+G increases their combined score to 100% (50% for each page).
2. Score redistribution:
   * When the surfer favors Page A the main cluster (A-E) also gains higher scoring.
   * When the surfer favors F+G the spider trap dominates despite being isolated.
   * The dangling node H is relatively stable across scenarios.
3. Practical Applications:
   * Based on the users' browsing history, we can apply personalized search.
4. Mathematical insights:
   * Personal bias changes the dominant eigenvector.
   * Different teleportation = different steady-state.
   * Link structure still matters (authority still propagates).


## Network Connectivity

PageRank depends heavily on the web structure. Adjusting the links can isolate parts of the graph or affects "authority propagation". With this experiment hereby we will remove several critical links and observe the impact. Based on our current web structure we will do the following:    

1. Remove the link D → E: This will remove the bridge from main cluster to the spider trap path.
2. Remove the link C → A: This will createa a cycle in main cluster within the graph.
3. Remove the link E → F: Remove the entry point to spider trap.

## Link Manipulation

We will investigate how strong is the PageRank algorithm against manipulation "attacks". Websites' Owners can try to artificially improve their scores creating fake links pointing to their site, getting links from pages with high scores or use other linking schemes.

In this experiment we will simulate an attacker who tries to boost Page # score (low score) through strategic link addition: 

1. Add link from Page # (page with the highest score) to Page #
2. Add links from pages #, # and # (multiple high score pages) to page #
3. Same as point 2, but test lower lower damping factor.   
